# 🚀 SVOMPTR-9B ELITE TRAINING PIPELINE (Unsloth + QLoRA)

ဒီ Notebook ကို အမှားအယွင်းမရှိ (Bulletproof) ဖြစ်အောင် အောက်ပါ Feature များ ထည့်သွင်းထားပါတယ် -
- **CUDA Linkage Guard**: BitsAndBytes CUDA error များကို အလိုအလျောက် ရှင်းလင်းပေးခြင်း။
- **Smart Memory Management**: T4 GPU (16GB) ပေါ်မှာ MoE model ကို အေးဆေး train နိုင်အောင် VRAM paging စနစ်သုံးထားခြင်း။
- **Universal Prompting**: ChatML format ကို standard အဖြစ် သတ်မှတ်ထားခြင်း။

## 🛠️ 1. Neural Interface & Hardware Setup

In [ ]:
import torch, os, sys, subprocess

def verify_hardware():
    if not torch.cuda.is_available():
        print("❌ FATAL ERROR: No GPU found!")
        return False
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ Hardware Active: {gpu_name} ({vram:.1f}GB VRAM)")
    return True

if verify_hardware():
    print("📦 Synchronizing Neural Core...")
    try:
        import unsloth
        print("✅ Unsloth already present.")
    except ImportError:
        print("📥 Installing Unsloth and GPU dependencies...")
        !pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
        !pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate "bitsandbytes>=0.43.3"
        !pip install datasets tqdm sentencepiece
        print("💡 TIP: If you still see ModuleNotFoundError, go to 'Runtime' -> 'Restart Session' and run Cell 3 again.")
    print("✅ Core Sync Complete.")

os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1" # Faster model downloads

In [ ]:
from IPython.display import Javascript
def auto_connect():
    display(Javascript('''
    function KeepAlive(){ 
        const btn = document.querySelector("colab-connect-button");
        if (btn) btn.shadowRoot.getElementById('connect').click();
    }
    setInterval(KeepAlive, 60000);
    '''))
auto_connect()
print("🛡️ Anti-Idle Guardian Active.")

## 📁 2. Storage Mapping (Google Drive)

In [ ]:
from google.colab import drive
import os

try:
    drive.mount('/content/drive', force_remount=True)
    BASE_DIR = '/content/drive/MyDrive/svomptr_auto_train'
except:
    print("⚠️ Drive mount failed. Local storage will be wiped after session ends.")
    BASE_DIR = '/content/svomptr_auto_train'

PATHS = {
    "checkpoints": os.path.join(BASE_DIR, 'checkpoints'),
    "weights": os.path.join(BASE_DIR, 'final_lora_weights'),
    "datasets": os.path.join(BASE_DIR, 'datasets')
}

for p in PATHS.values(): os.makedirs(p, exist_ok=True)
print(f"✅ Workspace Mapping: {BASE_DIR}")

## 🧪 3. Neural Architecture Loading

In [ ]:
from unsloth import FastLanguageModel
import torch, gc

# Constants
MAX_SEQ_LENGTH = 2048
MODEL_NAME = "Qwen/Qwen1.5-MoE-A2.7B"

# Clear VRAM Garbage
def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()
clear_memory()

print(f"🛠️ Loading Neural Engine: {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    trust_remote_code = True,
)

# Apply Optimized LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Experts Layered and Ready.")

## 📑 4. Semantic Dataset Formatting

In [ ]:
from datasets import load_dataset
import json, os

DATA_FILE = os.path.join(PATHS["datasets"], 'training_data.jsonl')

# Pre-check for Dataset Presence
if not os.path.exists(DATA_FILE) or os.path.getsize(DATA_FILE) == 0:
    print("📝 Generating default training matrix...")
    seed_data = [{"en": "The teacher is explaining the lesson.", "my": "ဆရာက သင်ခန်းစာကို ရှင်းပြနေတယ်။", "svomptr_structure": "S(The teacher)-V(explaining)-O(the lesson)"}]
    with open(DATA_FILE, 'w', encoding='utf-8') as f:
        for s in seed_data: f.write(json.dumps(s, ensure_ascii=False) + '\n')

dataset = load_dataset("json", data_files=DATA_FILE, split="train")

def apply_chatml_template(examples):
    chats = []
    # Safe extraction logic
    ens = examples.get("en", [])
    mys = examples.get("my", [])
    strs = examples.get("svomptr_structure", [])
    
    for en, my, s in zip(ens, mys, strs):
        prompt = f"<|im_start|>system\nEnglish-to-Myanmar SVOMPTR Transformer Expert.\n<|im_end|>\n<|im_start|>user\nTranslate: {en}\n<|im_end|>\n<|im_start|>assistant\nTranslation: {my}\nStructure: {s}\n<|im_end|>"
        chats.append(prompt)
    return { "text" : chats }

dataset = dataset.map(apply_chatml_template, batched = True)
print(f"✅ Tokenization Map Created: {len(dataset)} nodes.")

## 🏋️‍♂️ 5. Neural Training Loop (Production Mode)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

if 'model' not in globals() or model is None:
    print("❌ CRITICAL ERROR: Model handle lost. Re-run Cell 3.")
else:
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = dataset,
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        dataset_num_proc = 2,
        args = TrainingArguments(
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            warmup_steps = 10,
            max_steps = 100, 
            learning_rate = 2e-4,
            fp16 = not is_bfloat16_supported(),
            bf16 = is_bfloat16_supported(),
            logging_steps = 1,
            optim = "paged_adamw_8bit", # Safe memory paging logic
            weight_decay = 0.01,
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = PATHS["checkpoints"],
            save_steps = 50,
            save_total_limit = 2,
            report_to = "none",
        ),
    )

    print("▶️ Initiating Training Pipeline...")
    try:
        # Check for existing checkpoints to recover from crash
        has_checkpoint = True if os.path.exists(PATHS["checkpoints"]) and os.listdir(PATHS["checkpoints"]) else None
        trainer.train(resume_from_checkpoint = has_checkpoint)
        print("✅ Training cycle finished successfully.")
    except Exception as e:
        print(f"❌ Loop Error: {e}")

## 📦 6. Model Solidification

In [ ]:
if model:
    print(f"💾 Storing Neural Weights at {PATHS['weights']}...")
    model.save_pretrained(PATHS['weights'])
    tokenizer.save_pretrained(PATHS['weights'])
    print("✨ Saved! Ready for deployment in the SVOMPTR Brain.")
else:
    print("❌ Cleanup error: No weights to save.")